In [6]:
!pip install pandas indoNLP nlp-id Sastrawi

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 2.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.9/121.9 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.1/8.1 MB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 561.5/561.5 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.3/37.3 MB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.7/209.7 kB 2.0 MB/s eta 0:00:00
  Created wheel for wget: filename=wget-3.2-py3-none-any.whl size=9655 sha256=0052286b8c18f8a10b24ef03df722c283b9c2eea343ed3bccecbdefa9a2c3364
  Stored in directory: /root/.cache/pip/wheels/01/46/3b/e29ffbe4ebe614ff224bad40fc6a5773a67a163251585a13a9
Successfully built wget
  Attempting uninstall: scipy
    Found existing installation: scipy 1.16.3
    Uninstalling scipy-1.16.3:
      Successfully uninstall

In [8]:
import pandas as pd
import re
import collections

# Import library ajaib untuk NLP Bahasa Indonesia
from indoNLP.preprocessing import replace_slang, replace_word_elongation
from nlp_id.stopword import StopWord
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

# ==========================================
# 1. INISIALISASI TOOLS
# ==========================================
print("Menyiapkan kamus Stopword dan Stemmer...")
stopword = StopWord()
stemmer = StemmerFactory().create_stemmer()
print("Tools NLP siap digunakan!")

Menyiapkan kamus Stopword dan Stemmer...
Tools NLP siap digunakan!


In [9]:
# ==========================================
# 2. FUNGSI PREPROCESSING SUPER LENGKAP
# ==========================================
def text_preprocessing_pipeline(text):
    # Jaga-jaga kalau ada data kosong/bukan teks
    if not isinstance(text, str):
        return ""

    # A. Basic Cleaning (Regex)
    text = text.lower()
    text = re.sub(r'@[A-Za-z0-9_]+', ' ', text)  # Hapus mention IG
    text = re.sub(r'#[A-Za-z0-9_]+', ' ', text)  # Hapus hashtag
    text = re.sub(r'http\S+|www\S+', ' ', text)  # Hapus link/URL
    text = re.sub(r'[^a-z\s]', ' ', text)        # Sisakan huruf saja

    # B. Normalisasi Alay (indoNLP)
    # 1. Benerin huruf panjang (misal: "bengissss" -> "bengis")
    text = replace_word_elongation(text)
    # 2. Benerin kata gaul/singkatan (misal: "bgt" -> "banget", "lu" -> "kamu")
    text = replace_slang(text)

    # C. Hapus Stopword (nlp-id)
    text = stopword.remove_stopword(text)

    # D. Stemming (Sastrawi) - Mengembalikan ke kata dasar
    text = stemmer.stem(text)

    return text.strip()

In [10]:
# ==========================================
# 3. LOAD DATA & EKSEKUSI PREPROCESSING DAN TOKENISASI
# ==========================================
nama_file = 'DATASET CYBERBULLYING INSTAGRAM - FINAL.xlsx' # Ganti sesuai nama file Excel kamu

try:
    print(f"\nMemuat data dari {nama_file}...")
    df = pd.read_excel(nama_file)

    # Eksekusi fungsi preprocessing ke dalam kolom baru
    print("Memulai proses pembersihan teks..")
    df['Teks_Bersih'] = df['Komentar'].apply(text_preprocessing_pipeline)

    # ==========================================
    # 4. TOKENIZATION
    # ==========================================
    # Memecah kalimat yang sudah bersih menjadi list kata (token)
    df['Tokens'] = df['Teks_Bersih'].apply(lambda x: x.split())

    # Hapus baris yang jadi kosong melompong setelah dibersihkan
    df = df[df['Teks_Bersih'] != ''].reset_index(drop=True)

    print("Preprocessing & Tokenization Selesai! 🎉\n")

    # Preview 10 data pertama untuk melihat perubahannya
    print("Preview Hasil:")
    display(df[['Komentar', 'Teks_Bersih', 'Tokens']].head(10))

except FileNotFoundError:
    print(f"Ups! File '{nama_file}' nggak ketemu. Pastikan filenya satu folder sama kode ini ya.")


Memuat data dari DATASET CYBERBULLYING INSTAGRAM - FINAL.xlsx...
Memulai proses pembersihan teks..
Preprocessing & Tokenization Selesai! 🎉

Preview Hasil:


,Komentar,Teks_Bersih,Tokens
0,"""Kaka tidur yaa, udah pagi, gaboleh capek2""",kakak tidur pagi gaboleh capek,"[kakak, tidur, pagi, gaboleh, capek]"
1,"""makan nasi padang aja begini badannya""",makan nasi padang badan,"[makan, nasi, padang, badan]"
2,"""yang aku suka dari dia adalah selalu cukur je...",suka cukur jembut manggung,"[suka, cukur, jembut, manggung]"
3,"""Hai kak Isyana aku ngefans banget sama kak Is...",hai kak isyana fan kak isyana suka lagu kak is...,"[hai, kak, isyana, fan, kak, isyana, suka, lag..."
4,"""Manusia apa bidadari sih herann deh cantik te...",manusia bidadari heran deh cantik,"[manusia, bidadari, heran, deh, cantik]"
5,"""@ayu.kinantii isyan skrg berubah ya:( baju ny...",kinanti isyan ubah baju nakal,"[kinanti, isyan, ubah, baju, nakal]"
6,"""Gemesnya isyan kayak tango, berlapis lapis ci...",gemas isyan kayak tango lap lapis cia,"[gemas, isyan, kayak, tango, lap, lapis, cia]"
7,"""Makin jelek aja anaknya, padahal ibu ayahnya ...",jelek anak ayah cakep,"[jelek, anak, ayah, cakep]"
8,"""Kok anaknya kayak udah tua gitu ya mukanya kk...",anak kayak tua muka tasya,"[anak, kayak, tua, muka, tasya]"
9,"""Muka anak nya ko tua banget yaa.. GK ngegemes...",muka anak tua enggak gemas enggak lucu,"[muka, anak, tua, enggak, gemas, enggak, lucu]"


In [11]:
# ==========================================
# 5. SIMPAN HASIL (Opsional)
# ==========================================
df.to_csv('dataset_final_2.csv', index=True)

In [12]:
# ==========================================
# 6. ANALISIS DATA SEDERHANA
# ==========================================

# 1. Distribusi Kelas (Kategori)
print("\n=== Distribusi Kategori ===")
print(df['Kategori'].value_counts())
print("\n")

# 2. Top 10 Kata Paling Sering Muncul
print("=== Top 10 Kata Paling Sering Muncul ===")
# Gabungkan semua token menjadi satu list
all_words = [word for tokens_list in df['Tokens'] for word in tokens_list]

# Hitung frekuensi kata
word_counts = collections.Counter(all_words)

# Tampilkan 10 kata teratas
print(word_counts.most_common(10))


=== Distribusi Kategori ===
Kategori
Non-bullying    325
Bullying        325
Name: count, dtype: int64


=== Top 10 Kata Paling Sering Muncul ===
[('enggak', 149), ('kayak', 105), ('lu', 82), ('cantik', 81), ('muka', 65), ('anjing', 65), ('orang', 60), ('lo', 55), ('kak', 50), ('kalo', 36)]


In [14]:
# ==========================================
# 5. SIMPAN HASIL
# ==========================================
df.to_csv('dataset_final_2.csv', index=True)